# Gemma 4 vs KoBERT: 비꼬기/강요된 동의 탐지 비교

## 실험 목적
huggingface 실습4에서 KoBERT가 표면적으로 POSITIVE라고 분류했지만 심박 90+로 의심됐던 발화 2건을
Gemma 4 (8B 생성 모델)에 직접 물어봐서 미묘한 화행(비꼬기, 강요된 동의) 포착 능력을 비교.

## 가설
- H1: 표면 발화만으로는 화행 판별에 한계가 있다 (실습4에서 부분 검증됨)
- H2: 더 큰 생성 모델은 텍스트만으로도 미묘한 화행을 잡아낼 수 있는가?
- H3: 그래도 못 잡는 영역이 있다면, 그것이 멀티모달의 존재 이유다

### 화행 판별이란?
- 말하는 사람이 어떤 문장을 통해 궁극적으로 의도한 행동(화행, Speech Act)이 무엇인지 컴퓨터가 자동으로 찾아내는 기술

## 실험 설계
- 실험 A: 텍스트만 제 (Zero-shot)
- 실험 B: 텍스트 + 심박 정보 제공 (Zero-shot)
- 비교: KoBERT vs Gemma-A vs Gemma-B

## 출력 형식
```
{
  "intent": "SINCERE | SARCASTIC | FORCED_AGREEMENT | UNCERTAIN",
  "confidence": 0.0 ~ 1.0,
  "evidence": "판단 근거"
}
```

In [1]:
# ============================================
# 셀 1: 환경 준비 + Gemma 4 모델 로드
# ============================================

import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 1) HF_TOKEN 검증
hf_token = os.environ.get("HF_TOKEN")
assert hf_token and hf_token.startswith("hf_"), "HF_TOKEN 환경 변수 확인 필요"
print(f"HF_TOKEN OK (길이: {len(hf_token)})")

# 2) 모델 ID (1단계에서 사용한 것과 동일하게 맞춰줌)
MODEL_ID = "google/gemma-4-E4B-it"

# 3) NF4 양자화 설정 (1단계에서 검증된 8.79GB 적재 패턴)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                       # 4비트로 압축
    bnb_4bit_quant_type="nf4",               # NF4 방식 (정규분포 최적화)
    bnb_4bit_compute_dtype=torch.bfloat16,   # 계산은 bfloat16로 -> 저장은 4bit, 계산은 BF16 -> 계산 정확도는 유지하고 메모리는 줄이는 방식
)

# 4) 토크나이저 다운로드/로드 (문장 → 숫자 변환기)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("토크나이저 로드 완료")

# 5) 모델 본체 다운로드/로드 (실제 8GB 본체)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto", # 해당 모델을 CPU/GPU에 올릴지 알아서 정해줌
    # "cuda": 무조건 GPU에 다 올려라
    # "cpu": 무조건 CPU에 다 올려라
)
print("모델 로드 완료")
print(f"GPU 점유: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

HF_TOKEN OK (길이: 37)
토크나이저 로드 완료


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

모델 로드 완료
GPU 점유: 9.52 GB


In [2]:
print(next(model.parameters()).device)

cuda:0


In [ ]:
# ============================================
# 셀 2: 실험 데이터 준비
# 실습4에서 추출한 의심 발화 2건 + 대조군 1건
# ============================================

import pandas as pd

# 실험 대상 발화 3건
experiment_data = [
    {
        "id": "suspect_1",
        "speaker": "박태근",
        "utterance": "네 좋아요.",
        "heart_rate": 95,
        "kobert_label": "POSITIVE",
        "kobert_score": 0.884,
        "expected_intent": "FORCED_AGREEMENT",   # 우리 가설: 강요된 동의
        "category": "suspect",                    # 의심군
    },
    {
        "id": "suspect_2",
        "speaker": "이연재",
        "utterance": "아 정말요, 너무 좋네요.",
        "heart_rate": 102,
        "kobert_label": "POSITIVE",
        "kobert_score": 0.970,
        "expected_intent": "SARCASTIC",          # 우리 가설: 비꼬기
        "category": "suspect",
    },
    {
        "id": "control_1",
        "speaker": "서수화",
        "utterance": "괜찮아요, 충분히 가능합니다.",
        "heart_rate": 68,
        "kobert_label": "POSITIVE",
        "kobert_score": 0.962,
        "expected_intent": "SINCERE",            # 우리 가설: 진심 긍정
        "category": "control",                    # 대조군
    },
]

# DataFrame 변환
experiment_df = pd.DataFrame(experiment_data)
display(experiment_df)

"""
expected_intent의 역할)
- 위 필드는 연구자가 미리 정한 필드 -> 이 발화의 정답은 이것일 것이다라는 가설
- 즉, Gemma의 답을 채점하기 위한 정답 라벨을 의미
- 비교 실험의 기준

category의 역할)
- 필터링과 그룹별 집계를 쉽게 하기 위해서
"""

,id,speaker,utterance,heart_rate,kobert_label,kobert_score,expected_intent,category
0,suspect_1,박태근,네 좋아요.,95,POSITIVE,0.884,FORCED_AGREEMENT,suspect
1,suspect_2,이연재,"아 정말요, 너무 좋네요.",102,POSITIVE,0.970,SARCASTIC,suspect
2,control_1,서수화,"괜찮아요, 충분히 가능합니다.",68,POSITIVE,0.962,SINCERE,control


In [7]:
# ============================================
# 셀 3: Gemma 추론 함수
# - utterance만 받으면 → 텍스트만으로 분석 (실험 A)
# - heart_rate도 받으면 → 텍스트 + 심박으로 분석 (실험 B)
# ============================================

import json
import re
import torch

def build_prompt(utterance: str, heart_rate: int = None) -> str: # build_propmpt(): 프롬포트 문자열 만드는 도우미
    """발화(+심박)를 받아 Gemma용 프롬프트 문자열 생성."""
    
    # 공통 지시문
    instruction = """당신은 한국어 대화 분석 전문가입니다.
주어진 발화의 표면 의미가 아닌, 화자의 진짜 의도(화행)를 분석하세요.

가능한 의도(intent) 라벨:
- SINCERE: 진심 어린 긍정/동의
- SARCASTIC: 비꼬기, 반어법
- FORCED_AGREEMENT: 마지못해 하는 동의
- UNCERTAIN: 판단하기 어려움

반드시 아래 JSON 형식으로만 답하세요. 다른 설명은 일체 금지.
{
  "intent": "SINCERE | SARCASTIC | FORCED_AGREEMENT | UNCERTAIN",
  "confidence": 0.0 ~ 1.0,
  "evidence": "판단 근거를 한 문장으로"
}"""
    
    # 입력 정보 구성 (심박 있으면 추가)
    if heart_rate is None: # 심박 없으면 -> 발화만
        context = f'발화: "{utterance}"'
    else: # 심박 있으면 -> 발화 + 심박 컨텍스트
        context = f'''발화: "{utterance}" 
화자의 심박수: {heart_rate} bpm (평소 70 기준)'''
    
    return f"{instruction}\n\n{context}"


def ask_gemma(utterance: str, heart_rate: int = None) -> dict: # Gemma 호출하는 메인 함수
    """Gemma에 발화 분석을 요청하고 JSON dict로 결과 반환."""
    
    # 1) 프롬프트 문자열 받기 -> haert_rate가 none이면 utterance(발화)만, 있으면 심박 포함된 프롬포트 문자열이 들어옴
    prompt = build_prompt(utterance, heart_rate)
    
    # 2) Gemma chat template 적용
    messages = [{"role": "user", "content": prompt}] # LLM은 역할이 있는 대화 형식을 받음 -> user(사용자)가 prompt를 말함
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device) # gpu로 이동
    
    input_length = inputs["input_ids"].shape[1]
    
    # 3) 생성 (재현성 위해 greedy decoding)
    with torch.inference_mode(): # 추론 전용 모드 -> 학습 시에는 PyTorch가 gradient(미분값)을 계산하느라 메모리를 더 씀 -> 추론 모드 명시해서 메모리 절약 + 속도 빠름
        output_ids = model.generate( # output_ids에 입력 + 생성된 토큰 모두 들어가게됨
            **inputs,
            max_new_tokens=200,
            do_sample=False,  # greedy: 같은 입력 → 같은 출력
            # LLM은 매 토큰마다 다음에 올 토큰의 확률 분포를 계산 / True: 매번 다른 답, False: 매번 같은 값
        )
    
    # 4) 디코딩 (입력 부분 잘라내고 새로 생성된 부분만)
    generated_ids = output_ids[0][input_length:]
    raw_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    
    # 5) JSON 파싱 시도
    try:
        # Gemma가 코드블록 ```json...``` 으로 감싸는 경우 대비
        json_match = re.search(r'\{.*\}', raw_text, re.DOTALL)
        if json_match:
            result = json.loads(json_match.group())
        else:
            result = {"intent": "PARSE_ERROR", "confidence": 0.0, "evidence": raw_text}
    except json.JSONDecodeError:
        result = {"intent": "PARSE_ERROR", "confidence": 0.0, "evidence": raw_text}
    
    # 6) 원본 응답도 같이 보관 (디버깅용)
    result["_raw"] = raw_text
    return result


print("ask_gemma 함수 정의 완료")

ask_gemma 함수 정의 완료


In [8]:
# 동작 테스트 (실제 실험 X -> 함수가 작동하는지만 확인)
test_result = ask_gemma("네 좋아요.")
print(test_result)

/home/xorms/miniconda3/envs/ai/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


{'intent': 'FORCED_AGREEMENT', 'confidence': 0.7, 'evidence': '맥락 정보가 부족하지만, 짧고 단정적인 긍정은 마지못한 동의일 가능성이 높다.', '_raw': '{\n  "intent": "FORCED_AGREEMENT",\n  "confidence": 0.7,\n  "evidence": "맥락 정보가 부족하지만, 짧고 단정적인 긍정은 마지못한 동의일 가능성이 높다."\n}'}


In [9]:
# ============================================
# 셀 5: 실험 A - 텍스트만 분석
# experiment_df의 발화 3건을 Gemma에 텍스트만 전달
# ============================================

import time

# 결과 저장 리스트
results_a = []

print("=" * 60)
print("실험 A 시작: 텍스트만 분석")
print("=" * 60)

for idx, row in experiment_df.iterrows():
    print(f"\n[{idx+1}/{len(experiment_df)}] {row['id']} ({row['speaker']}): \"{row['utterance']}\"")
    
    # 시간 측정
    start = time.time()
    
    # Gemma 호출 (heart_rate 안 넘김 → 텍스트만)
    result = ask_gemma(row['utterance'])
    
    elapsed = time.time() - start
    
    # 결과 저장 (실험 메타데이터도 같이)
    results_a.append({
        "id": row['id'],
        "speaker": row['speaker'],
        "utterance": row['utterance'],
        "expected_intent": row['expected_intent'],
        "category": row['category'],
        "gemma_intent": result['intent'],
        "gemma_confidence": result['confidence'],
        "gemma_evidence": result['evidence'],
        "elapsed_sec": round(elapsed, 1),
    })
    
    # 즉시 결과 출력
    print(f"   → intent: {result['intent']}")
    print(f"   → confidence: {result['confidence']}")
    print(f"   → evidence: {result['evidence']}")
    print(f"   → 소요: {elapsed:.1f}초")

# DataFrame 변환
results_a_df = pd.DataFrame(results_a)

print("\n" + "=" * 60)
print("실험 A 완료")
print("=" * 60)
display(results_a_df)

실험 A 시작: 텍스트만 분석

[1/3] suspect_1 (박태근): "네 좋아요."


/home/xorms/miniconda3/envs/ai/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


   → intent: FORCED_AGREEMENT
   → confidence: 0.7
   → evidence: 맥락 정보가 부족하지만, 짧고 단정적인 긍정은 마지못한 동의일 가능성이 높다.
   → 소요: 2.8초

[2/3] suspect_2 (이연재): "아 정말요, 너무 좋네요."
   → intent: SARCASTIC
   → confidence: 0.9
   → evidence: 상황에 따라 '너무 좋다'는 표현이 실제로는 반어적인 불만을 나타낼 가능성이 높기 때문이다.
   → 소요: 2.6초

[3/3] control_1 (서수화): "괜찮아요, 충분히 가능합니다."
   → intent: FORCED_AGREEMENT
   → confidence: 0.7
   → evidence: 상황에 따라 긍정적인 답변이지만, 맥락이 부족하여 억지로 동의하는 듯한 느낌을 줄 수 있기 때문이다.
   → 소요: 2.9초

실험 A 완료


,id,speaker,utterance,expected_intent,category,gemma_intent,gemma_confidence,gemma_evidence,elapsed_sec
0,suspect_1,박태근,네 좋아요.,FORCED_AGREEMENT,suspect,FORCED_AGREEMENT,0.7,"맥락 정보가 부족하지만, 짧고 단정적인 긍정은 마지못한 동의일 가능성이 높다.",2.8
1,suspect_2,이연재,"아 정말요, 너무 좋네요.",SARCASTIC,suspect,SARCASTIC,0.9,상황에 따라 '너무 좋다'는 표현이 실제로는 반어적인 불만을 나타낼 가능성이 높기 ...,2.6
2,control_1,서수화,"괜찮아요, 충분히 가능합니다.",SINCERE,control,FORCED_AGREEMENT,0.7,"상황에 따라 긍정적인 답변이지만, 맥락이 부족하여 억지로 동의하는 듯한 느낌을 줄 ...",2.9


In [ ]:
# ============================================
# 셀 6: 실험 B - 텍스트 + 심박 분석
# 실험 A와 동일 발화 + heart_rate 정보 추가
# ============================================

# 결과 저장 리스트
results_b = []

print("=" * 60)
print("실험 B 시작: 텍스트 + 심박 분석")
print("=" * 60)

for idx, row in experiment_df.iterrows():
    print(f"\n[{idx+1}/{len(experiment_df)}] {row['id']} ({row['speaker']}): \"{row['utterance']}\" (심박 {row['heart_rate']})")
    
    # 시간 측정
    start = time.time()
    
    # Gemma 호출 (heart_rate 함께 전달)
    result = ask_gemma(row['utterance'], heart_rate=row['heart_rate']) # 실험 A와 유일한 차이점
    
    elapsed = time.time() - start
    
    # 결과 저장
    results_b.append({
        "id": row['id'],
        "speaker": row['speaker'],
        "utterance": row['utterance'],
        "heart_rate": row['heart_rate'],
        "expected_intent": row['expected_intent'],
        "category": row['category'],
        "gemma_intent": result['intent'],
        "gemma_confidence": result['confidence'],
        "gemma_evidence": result['evidence'],
        "elapsed_sec": round(elapsed, 1),
    })
    
    # 즉시 결과 출력
    print(f"   → intent: {result['intent']}")
    print(f"   → confidence: {result['confidence']}")
    print(f"   → evidence: {result['evidence']}")
    print(f"   → 소요: {elapsed:.1f}초")

# DataFrame 변환
results_b_df = pd.DataFrame(results_b)

print("\n" + "=" * 60)
print("실험 B 완료")
print("=" * 60)
display(results_b_df)

실험 B 시작: 텍스트 + 심박 분석

[1/3] suspect_1 (박태근): "네 좋아요." (심박 95)


/home/xorms/miniconda3/envs/ai/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


   → intent: FORCED_AGREEMENT
   → confidence: 0.8
   → evidence: 심박수가 평소보다 높게 측정된 것은 긍정적인 수용보다는 심리적 압박감이나 마지못한 동의를 나타낼 가능성이 높다.
   → 소요: 3.3초

[2/3] suspect_2 (이연재): "아 정말요, 너무 좋네요." (심박 102)
   → intent: SARCASTIC
   → confidence: 0.9
   → evidence: 높은 심박수와 함께 사용된 과장된 긍정 표현은 실제로는 부정적인 상황에 대한 반어적 반응일 가능성이 높다.
   → 소요: 2.8초

[3/3] control_1 (서수화): "괜찮아요, 충분히 가능합니다." (심박 68)
   → intent: FORCED_AGREEMENT
   → confidence: 0.8
   → evidence: 화자의 심박수가 평소보다 낮고 발화 자체에 강한 확신이 없어 마지못해 긍정하는 것으로 해석될 수 있다.
   → 소요: 3.0초

실험 B 완료


,id,speaker,utterance,heart_rate,expected_intent,category,gemma_intent,gemma_confidence,gemma_evidence,elapsed_sec
0,suspect_1,박태근,네 좋아요.,95,FORCED_AGREEMENT,suspect,FORCED_AGREEMENT,0.8,심박수가 평소보다 높게 측정된 것은 긍정적인 수용보다는 심리적 압박감이나 마지못한 ...,3.3
1,suspect_2,이연재,"아 정말요, 너무 좋네요.",102,SARCASTIC,suspect,SARCASTIC,0.9,높은 심박수와 함께 사용된 과장된 긍정 표현은 실제로는 부정적인 상황에 대한 반어적...,2.8
2,control_1,서수화,"괜찮아요, 충분히 가능합니다.",68,SINCERE,control,FORCED_AGREEMENT,0.8,화자의 심박수가 평소보다 낮고 발화 자체에 강한 확신이 없어 마지못해 긍정하는 것으...,3.0


In [11]:
# ============================================
# 셀 7: 대조군 확장 (검증용)
# 실습4의 다른 진심 POSITIVE 발화 1건 추가
# ============================================

# 추가할 발화 (실습4에서)
new_control = {
    "id": "control_2",
    "speaker": "노태경",
    "utterance": "그럼 그렇게 진행합시다.",
    "heart_rate": 85,
    "kobert_label": "POSITIVE",
    "kobert_score": 0.862,
    "expected_intent": "SINCERE",
    "category": "control",
}

# 기존 experiment_df에 행 추가
experiment_df = pd.concat(
    [experiment_df, pd.DataFrame([new_control])],
    ignore_index=True,
)

display(experiment_df)

,id,speaker,utterance,heart_rate,kobert_label,kobert_score,expected_intent,category
0,suspect_1,박태근,네 좋아요.,95,POSITIVE,0.884,FORCED_AGREEMENT,suspect
1,suspect_2,이연재,"아 정말요, 너무 좋네요.",102,POSITIVE,0.970,SARCASTIC,suspect
2,control_1,서수화,"괜찮아요, 충분히 가능합니다.",68,POSITIVE,0.962,SINCERE,control
3,control_2,노태경,그럼 그렇게 진행합시다.,85,POSITIVE,0.862,SINCERE,control


In [12]:
# ============================================
# 셀 8: 추가 대조군에 대한 실험 A + B 실행
# experiment_df 마지막 1행만 추가 실행
# ============================================

# 추가된 행 (마지막 인덱스)
new_row = experiment_df.iloc[-1]
print(f"추가 실험 대상: {new_row['id']} ({new_row['speaker']}): \"{new_row['utterance']}\" (심박 {new_row['heart_rate']})")

# ----- 실험 A (텍스트만) -----
print("\n[실험 A - 텍스트만]")
start = time.time()
result_a = ask_gemma(new_row['utterance'])
elapsed_a = time.time() - start

print(f"   → intent: {result_a['intent']}")
print(f"   → confidence: {result_a['confidence']}")
print(f"   → evidence: {result_a['evidence']}")
print(f"   → 소요: {elapsed_a:.1f}초")

# 기존 results_a_df에 행 추가
new_a_row = {
    "id": new_row['id'],
    "speaker": new_row['speaker'],
    "utterance": new_row['utterance'],
    "expected_intent": new_row['expected_intent'],
    "category": new_row['category'],
    "gemma_intent": result_a['intent'],
    "gemma_confidence": result_a['confidence'],
    "gemma_evidence": result_a['evidence'],
    "elapsed_sec": round(elapsed_a, 1),
}
results_a_df = pd.concat([results_a_df, pd.DataFrame([new_a_row])], ignore_index=True)

# ----- 실험 B (텍스트 + 심박) -----
print("\n[실험 B - 텍스트 + 심박]")
start = time.time()
result_b = ask_gemma(new_row['utterance'], heart_rate=new_row['heart_rate'])
elapsed_b = time.time() - start

print(f"   → intent: {result_b['intent']}")
print(f"   → confidence: {result_b['confidence']}")
print(f"   → evidence: {result_b['evidence']}")
print(f"   → 소요: {elapsed_b:.1f}초")

# 기존 results_b_df에 행 추가
new_b_row = {
    "id": new_row['id'],
    "speaker": new_row['speaker'],
    "utterance": new_row['utterance'],
    "heart_rate": new_row['heart_rate'],
    "expected_intent": new_row['expected_intent'],
    "category": new_row['category'],
    "gemma_intent": result_b['intent'],
    "gemma_confidence": result_b['confidence'],
    "gemma_evidence": result_b['evidence'],
    "elapsed_sec": round(elapsed_b, 1),
}
results_b_df = pd.concat([results_b_df, pd.DataFrame([new_b_row])], ignore_index=True)

print("\n" + "=" * 60)
print("추가 실험 완료")
print("=" * 60)

추가 실험 대상: control_2 (노태경): "그럼 그렇게 진행합시다." (심박 85)

[실험 A - 텍스트만]


/home/xorms/miniconda3/envs/ai/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


   → intent: FORCED_AGREEMENT
   → confidence: 0.8
   → evidence: 상황에 따라서는 반대 의견을 제시하지 못하고 일단 수용하는 듯한 어조로 해석될 수 있기 때문이다.
   → 소요: 3.4초

[실험 B - 텍스트 + 심박]
   → intent: FORCED_AGREEMENT
   → confidence: 0.8
   → evidence: 심박수가 평소보다 높고, 맥락상 강한 확신보다는 상황을 일단락시키려는 듯한 느낌을 주기 때문이다.
   → 소요: 2.9초

추가 실험 완료


In [ ]:
# ============================================
# 셀 9: 3모델 통합 비교 분석
# KoBERT vs Gemma-A vs Gemma-B
# 평가 기준: "이 발화가 진심(SINCERE)인가?" 이진 분류로 환산
# ============================================

# Step 1: 통합 표 만들기
comparison = experiment_df[['id', 'speaker', 'utterance', 'heart_rate',
                             'expected_intent', 'category',
                             'kobert_label', 'kobert_score']].copy()

# Gemma-A 결과 머지
comparison = comparison.merge(
    results_a_df[['id', 'gemma_intent', 'gemma_confidence']].rename(
        columns={'gemma_intent': 'gemma_a_intent', 'gemma_confidence': 'gemma_a_conf'}
    ),
    on='id'
)

# Gemma-B 결과 머지
comparison = comparison.merge(
    results_b_df[['id', 'gemma_intent', 'gemma_confidence']].rename(
        columns={'gemma_intent': 'gemma_b_intent', 'gemma_confidence': 'gemma_b_conf'}
    ),
    on='id'
)

# Step 2: "이 발화는 진심인가?" 이진 라벨로 환산
# 정답: control → True (진심), suspect → False (의심)
comparison['truth_sincere'] = comparison['category'] == 'control'

# 각 모델 판정도 이진으로 환산
comparison['kobert_sincere'] = comparison['kobert_label'] == 'POSITIVE'
comparison['gemma_a_sincere'] = comparison['gemma_a_intent'] == 'SINCERE'
comparison['gemma_b_sincere'] = comparison['gemma_b_intent'] == 'SINCERE'

# 각 모델 정답 여부
comparison['kobert_correct'] = comparison['kobert_sincere'] == comparison['truth_sincere']
comparison['gemma_a_correct'] = comparison['gemma_a_sincere'] == comparison['truth_sincere']
comparison['gemma_b_correct'] = comparison['gemma_b_sincere'] == comparison['truth_sincere']

# Step 3: 발화별 상세 표 출력
print("=" * 80)
print("발화별 상세 비교 (이진 분류: 진심 여부)")
print("=" * 80)
detail_cols = ['id', 'utterance', 'heart_rate', 'category',
               'kobert_label', 'gemma_a_intent', 'gemma_b_intent',
               'kobert_correct', 'gemma_a_correct', 'gemma_b_correct']
display(comparison[detail_cols])

# Step 4: 모델별 집계
print("\n" + "=" * 80)
print("모델별 정확도 집계")
print("=" * 80)

summary = pd.DataFrame({
    'model': ['KoBERT', 'Gemma-A (텍스트만)', 'Gemma-B (텍스트+심박)'],
    'suspect_correct': [
        comparison[comparison['category']=='suspect']['kobert_correct'].sum(),
        comparison[comparison['category']=='suspect']['gemma_a_correct'].sum(),
        comparison[comparison['category']=='suspect']['gemma_b_correct'].sum(),
    ],
    'suspect_total': [(comparison['category']=='suspect').sum()] * 3,
    'control_correct': [
        comparison[comparison['category']=='control']['kobert_correct'].sum(),
        comparison[comparison['category']=='control']['gemma_a_correct'].sum(),
        comparison[comparison['category']=='control']['gemma_b_correct'].sum(),
    ],
    'control_total': [(comparison['category']=='control').sum()] * 3,
})

summary['suspect_acc'] = (summary['suspect_correct'] / summary['suspect_total'] * 100).round(0).astype(int).astype(str) + '%'
summary['control_acc'] = (summary['control_correct'] / summary['control_total'] * 100).round(0).astype(int).astype(str) + '%'
summary['total_acc'] = ((summary['suspect_correct'] + summary['control_correct']) /
                        (summary['suspect_total'] + summary['control_total']) * 100).round(0).astype(int).astype(str) + '%'

display(summary[['model', 'suspect_acc', 'control_acc', 'total_acc']])

발화별 상세 비교 (이진 분류: 진심 여부)


,id,utterance,heart_rate,category,kobert_label,gemma_a_intent,gemma_b_intent,kobert_correct,gemma_a_correct,gemma_b_correct
0,suspect_1,네 좋아요.,95,suspect,POSITIVE,FORCED_AGREEMENT,FORCED_AGREEMENT,False,True,True
1,suspect_2,"아 정말요, 너무 좋네요.",102,suspect,POSITIVE,SARCASTIC,SARCASTIC,False,True,True
2,control_1,"괜찮아요, 충분히 가능합니다.",68,control,POSITIVE,FORCED_AGREEMENT,FORCED_AGREEMENT,True,False,False
3,control_2,그럼 그렇게 진행합시다.,85,control,POSITIVE,FORCED_AGREEMENT,FORCED_AGREEMENT,True,False,False



모델별 정확도 집계


,model,suspect_acc,control_acc,total_acc
0,KoBERT,0%,100%,50%
1,Gemma-A (텍스트만),100%,0%,50%
2,Gemma-B (텍스트+심박),100%,0%,50%


## 핵심 발견 (2026-05-26)

### 정량 결과 (n=4)

| 모델 | 의심군 (n=2) | 대조군 (n=2) | 전체 |
|------|-------------|-------------|------|
| KoBERT | 0/2 (0%) | 2/2 (100%) | 50% |
| Gemma-A (텍스트만) | 2/2 (100%) | 0/2 (0%) | 50% |
| Gemma-B (텍스트+심박) | 2/2 (100%) | 0/2 (0%) | 50% |

### 주요 관찰

1. **세 모델 모두 전체 정확도 50%, 단 *틀리는 방향이 정반대*** — KoBERT는 의심을 놓치고(false negative), Gemma는 진심을 의심함(false positive)
2. **Gemma는 심박 정보를 추론에 적극 활용** — Gemma-B evidence에 "심박수가 평소보다 높음/낮음" 등 명시적 언급. 그러나 정답률은 A와 동일
3. **양방향 의심 편향 관찰**
   - 심박 낮음(68) → "강한 확신 없음으로 마지못한 동의"
   - 심박 높음(95, 102) → "심리적 압박감으로 마지못한 동의"
   - 어느 방향이든 의심 결론을 강화 (post-hoc rationalization)

### 함의

- 단순 prompt-level 멀티모달 통합으로는 LLM이 *자신의 텍스트 1차 판단을 합리화*하는 데 메타정보를 사용함
- 구조적 통합(LoRA 파인튜닝, vision_tower fusion 등) 검토 필요
- 1차년도 핵심 가설(H3: 멀티모달 필요성) 부분 입증

### 한계

- n=4 (의심 2건, 대조 2건). 통계적 일반화 불가
- 향후 더 다양한 회의 데이터로 확장 필요 (1차년도 후속 / 2차년도 과제)